# Big Data com Google Colab
## Semana 1 — O que é Big Data e quais são os 5 Vs?

**Duração:** 3 horas  
**Conhecimento prévio:** noções básicas de computador; não é necessário dominar Python  
**Caso utilizado:** dados de mobilidade urbana

Este notebook combina a explicação da matéria com atividades de código totalmente resolvidas. O objetivo não é decorar comandos, mas compreender o raciocínio e aprender Python enquanto investigamos os dados.


## 1. Tema da aula

Todos os dias, celulares, sensores, lojas, hospitais, bancos, redes sociais e serviços públicos produzem dados. Parte desses dados pode ser analisada em uma planilha ou em um programa comum. Porém, alguns problemas apresentam uma combinação de características que torna o armazenamento e o processamento mais difíceis.

Chamamos esse contexto de **Big Data**.

Big Data não significa apenas “um arquivo muito grande”. Um problema pode exigir novas técnicas porque:

- existem dados demais para uma única máquina;
- os dados chegam rapidamente e precisam de resposta em pouco tempo;
- as informações vêm de muitos sistemas e formatos;
- parte dos registros está incompleta ou incorreta;
- é necessário transformar os dados em decisões úteis.

Essas características são normalmente organizadas nos **5 Vs de Big Data**: Volume, Velocidade, Variedade, Veracidade e Valor.


### Os 5 Vs

| V | Significado | Pergunta que devemos fazer |
|---|---|---|
| **Volume** | Quantidade de dados | Quanto precisamos armazenar e processar? |
| **Velocidade** | Ritmo de chegada e necessidade de resposta | Com que rapidez os dados chegam e quando precisamos do resultado? |
| **Variedade** | Diversidade de fontes, formatos e estruturas | De quantas formas diferentes os dados são produzidos? |
| **Veracidade** | Qualidade e confiabilidade | Podemos confiar nos registros? |
| **Valor** | Benefício produzido | Que decisão será melhorada com esses dados? |

```mermaid
flowchart LR
    A[Dados produzidos] --> B[Volume]
    B --> C[Velocidade]
    C --> D[Variedade]
    D --> E[Veracidade]
    E --> F[Valor]
    F --> G[Decisão]
```

Durante a aula, usaremos o mesmo conjunto de dados para observar cada V. Assim, a matéria e o código formarão uma única investigação.


## 2. O caso que estudaremos

Uma prefeitura recebe registros de ônibus, metrô, bicicletas e viagens por aplicativo. Os dados são produzidos por GPS, bilhetagem, estações e aplicativos.

A prefeitura quer descobrir:

- quais meios de transporte são mais utilizados;
- em quais horários existe maior movimento;
- quanto tempo as viagens costumam durar;
- se existem registros incorretos;
- se a infraestrutura suportará o crescimento futuro;
- como os dados podem melhorar o serviço público.

Cada pergunta está relacionada a um ou mais Vs. Antes de escolher ferramentas, precisamos entender o problema.

### Objetivos de aprendizagem

Ao final da aula, você deverá conseguir:

1. explicar o conceito de Big Data;
2. reconhecer os 5 Vs em um caso concreto;
3. medir o crescimento do volume de dados;
4. identificar problemas básicos de qualidade;
5. produzir indicadores simples;
6. compreender os comandos de Python utilizados nas atividades;
7. diferenciar uma execução local de um sistema distribuído.


## 3. Antes do laboratório — como ler o código Python

Não é necessário conhecer todo o Python. Nesta aula usaremos apenas alguns elementos:

| Elemento | Exemplo | Significado |
|---|---|---|
| Variável | `quantidade = 100` | Guarda um valor para utilizarmos depois |
| Texto | `cidade = "Recife"` | Textos ficam entre aspas |
| Lista | `["ônibus", "metrô"]` | Guarda vários valores em sequência |
| Função | `print(quantidade)` | Executa uma ação; neste caso, mostra um valor |
| DataFrame | `df` | Tabela formada por linhas e colunas |
| Coluna | `df["cidade"]` | Seleciona uma coluna da tabela |
| Método | `df.head()` | Executa uma operação pertencente ao DataFrame |

Leia uma expressão de dentro para fora. Por exemplo:

```python
df["cidade"].value_counts()
```

1. `df` é a tabela.
2. `df["cidade"]` seleciona a coluna cidade.
3. `.value_counts()` conta quantas vezes cada cidade aparece.

As células possuem comentários iniciados por `#`. O Python ignora esses comentários; eles existem para explicar o código.


## 4. Atividade — preparando o ambiente

Começaremos importando duas bibliotecas:

- **Pandas:** trabalha com tabelas chamadas DataFrames.
- **NumPy:** ajuda a criar números e amostras para nossa simulação.

`import` significa que desejamos usar uma biblioteca. `as` cria um apelido curto: `pd` para Pandas e `np` para NumPy.


In [ ]:
# Importa as bibliotecas e cria os apelidos pd e np.
import numpy as np
import pandas as pd

# display mostra tabelas de forma organizada no Colab.
from IPython.display import display

# Mostra as versões utilizadas na aula.
print("Versão do Pandas:", pd.__version__)
print("Versão do NumPy:", np.__version__)


## 5. Atividade — criando os dados da aula

Usaremos dados sintéticos: registros criados pelo próprio código para fins educacionais. Isso torna o notebook independente de downloads e serviços externos.

Primeiro definimos:

- a quantidade de eventos;
- uma semente aleatória, que faz todos receberem os mesmos valores;
- listas com cidades, modais, fontes e situações possíveis.

Depois o Pandas reúne as colunas em um DataFrame.


In [ ]:
# Letras maiúsculas indicam valores que não pretendemos alterar.
N_EVENTOS = 60_000
SEMENTE = 42

# O gerador cria valores aleatórios que podem ser repetidos em outra execução.
gerador = np.random.default_rng(SEMENTE)

# Listas com as opções que aparecerão nos dados.
cidades = ["São Paulo", "Recife", "Curitiba", "Goiânia"]
modais = ["ônibus", "metrô", "bicicleta", "aplicativo"]
fontes = ["bilhetagem", "gps", "aplicativo", "estação"]
status_possiveis = ["concluída", "cancelada", "interrompida"]

# Cria datas aleatórias ao longo de 31 dias.
inicio = pd.Timestamp("2026-03-01")
minutos_aleatorios = gerador.integers(0, 31 * 24 * 60, size=N_EVENTOS)
datas = inicio + pd.to_timedelta(minutos_aleatorios, unit="m")

# Cria distâncias e durações simples para cada evento.
distancias = gerador.uniform(1, 30, size=N_EVENTOS).round(2)
duracoes = (distancias * gerador.uniform(2, 5, size=N_EVENTOS)).round(1)

# Um dicionário associa o nome de cada coluna aos seus valores.
dados = {
    "evento_id": range(1, N_EVENTOS + 1),
    "data_hora": datas,
    "cidade": gerador.choice(cidades, size=N_EVENTOS),
    "modal": gerador.choice(modais, size=N_EVENTOS),
    "distancia_km": distancias,
    "duracao_min": duracoes,
    "status": gerador.choice(status_possiveis, size=N_EVENTOS),
    "fonte": gerador.choice(fontes, size=N_EVENTOS),
}

# DataFrame é a tabela criada pelo Pandas.
df_bruto = pd.DataFrame(dados)

# Mostra as cinco primeiras linhas.
df_bruto.head()


### Inserindo situações que existem em dados reais

Bases reais podem conter campos ausentes, erros de digitação, valores impossíveis e duplicidades. Vamos inserir alguns desses problemas para encontrá-los mais adiante.

`loc` seleciona linhas e colunas. `np.nan` representa um valor ausente. `pd.concat` junta tabelas.


In [ ]:
# Apaga a distância das primeiras 100 linhas.
df_bruto.loc[0:99, "distancia_km"] = np.nan

# Coloca durações impossíveis em algumas linhas.
df_bruto.loc[100:149, "duracao_min"] = -1
df_bruto.loc[150:179, "duracao_min"] = 999

# Simula uma grafia diferente para ônibus.
df_bruto.loc[200:299, "modal"] = "Onibus"

# Copia 200 linhas para simular registros duplicados.
linhas_repetidas = df_bruto.iloc[0:200].copy()
df_bruto = pd.concat([df_bruto, linhas_repetidas], ignore_index=True)

print("Quantidade final de linhas:", len(df_bruto))
display(df_bruto.head())


## 6. Volume — quanto dado existe?

**Volume** é a quantidade de dados que precisa ser armazenada e processada. Não devemos observar somente o número de linhas: textos, tipos das colunas e tempo de retenção também afetam o tamanho.

### Atividade — medindo linhas, colunas e memória

- `len(df_bruto)` conta as linhas.
- `df_bruto.shape[1]` informa a quantidade de colunas.
- `memory_usage()` calcula a memória utilizada.
- Dividimos por `1024 ** 2` para converter bytes em megabytes.


In [ ]:
# Conta as linhas da tabela.
quantidade_linhas = len(df_bruto)

# shape possui duas posições: linhas e colunas. A posição 1 representa colunas.
quantidade_colunas = df_bruto.shape[1]

# Soma a memória de todas as colunas e converte para megabytes.
memoria_bytes = df_bruto.memory_usage(deep=True).sum()
memoria_mb = memoria_bytes / 1024 ** 2

print("Linhas:", quantidade_linhas)
print("Colunas:", quantidade_colunas)
print("Memória aproximada em MB:", round(memoria_mb, 2))


### Atividade — observando o crescimento

Agora aparece nosso primeiro laço de repetição:

```python
for fator in [1, 2, 4, 8]:
```

O Python executará o bloco quatro vezes. Em cada repetição, `fator` receberá um valor da lista. Os resultados serão guardados em outra lista por meio de `.append()`.


In [ ]:
# Cria uma lista vazia para guardar os resultados.
resultados_volume = []

# Repete o mesmo teste para quatro tamanhos.
for fator in [1, 2, 4, 8]:
    # Repete a tabela de acordo com o fator atual.
    tabela_ampliada = pd.concat([df_bruto] * fator, ignore_index=True)

    # Mede a memória da nova tabela.
    memoria_atual = tabela_ampliada.memory_usage(deep=True).sum() / 1024 ** 2

    # Guarda fator, linhas e memória como um pequeno dicionário.
    resultado_atual = {
        "fator": fator,
        "linhas": len(tabela_ampliada),
        "memoria_mb": round(memoria_atual, 2),
    }
    resultados_volume.append(resultado_atual)

# Transforma a lista de resultados em uma tabela.
tabela_volume = pd.DataFrame(resultados_volume)
display(tabela_volume)


### O que aprendemos com o Volume?

A memória cresce aproximadamente junto com a quantidade de linhas. Em algum momento, uma única máquina pode deixar de ser suficiente.

- **Escala vertical:** aumentar memória e CPU da mesma máquina.
- **Escala horizontal:** dividir dados e trabalho entre várias máquinas.

Nesta aula, tudo continua rodando em uma única máquina do Colab. Repetir linhas não transforma a execução em Big Data; apenas ajuda a visualizar o crescimento.


## 7. Velocidade — com que rapidez os dados chegam?

**Velocidade** relaciona o ritmo de chegada dos dados ao tempo disponível para produzir uma resposta.

- Um relatório mensal pode ser calculado em lote.
- Um painel operacional pode precisar de atualização a cada cinco minutos.
- Um alerta pode precisar ser produzido imediatamente.

### Atividade — contando eventos por hora

`dt.hour` extrai somente a hora de uma data. `value_counts()` conta quantas vezes cada hora aparece. `sort_index()` coloca as horas em ordem.


In [ ]:
# Cria uma coluna contendo somente a hora de cada evento.
df_bruto["hora"] = df_bruto["data_hora"].dt.hour

# Conta os eventos de cada hora e organiza de 0 a 23.
eventos_por_hora = df_bruto["hora"].value_counts().sort_index()

# idxmax encontra a hora que possui o maior valor.
hora_pico = eventos_por_hora.idxmax()
quantidade_no_pico = eventos_por_hora.max()

print("Hora com mais eventos:", hora_pico)
print("Quantidade de eventos nessa hora:", quantidade_no_pico)
print()
print("Eventos por hora:")
display(eventos_por_hora.to_frame("quantidade"))


### O que aprendemos com a Velocidade?

Os mesmos eventos podem alimentar processos diferentes. A prefeitura pode produzir um relatório completo uma vez por dia e, ao mesmo tempo, atualizar um indicador operacional com maior frequência.

Não é necessário tornar tudo instantâneo. Primeiro definimos quanto atraso a decisão aceita; depois escolhemos a arquitetura.


## 8. Variedade — quantas formas de dados existem?

**Variedade** representa a diversidade de fontes, formatos, estruturas e significados.

Nossa tabela contém:

- números inteiros e decimais;
- textos e categorias;
- datas e horários;
- registros de quatro fontes diferentes.

Em um ambiente real, também poderíamos receber arquivos CSV, mensagens JSON, coordenadas de GPS, imagens e textos livres.

### Atividade — observando tipos e fontes

`dtypes` apresenta o tipo de cada coluna. `value_counts()` mostra a quantidade de registros produzidos por cada fonte.


In [ ]:
# Mostra o tipo utilizado para armazenar cada coluna.
print("Tipos das colunas:")
display(df_bruto.dtypes.to_frame("tipo"))

# Conta quantos registros vieram de cada fonte.
print("Registros por fonte:")
registros_por_fonte = df_bruto["fonte"].value_counts()
display(registros_por_fonte.to_frame("quantidade"))


### O que aprendemos com a Variedade?

Combinar fontes não significa apenas juntar colunas. É preciso descobrir se os campos possuem o mesmo significado, unidade, formato e nível de qualidade.

Um catálogo de dados registra essas informações e ajuda as pessoas a encontrarem e compreenderem os dados disponíveis.


## 9. Veracidade — podemos confiar nos dados?

**Veracidade** representa a qualidade e a confiabilidade. Um programa pode calcular uma resposta mesmo quando os registros estão errados. Por isso, resultado calculado não é necessariamente resultado confiável.

### Atividade — contando os problemas

Uma comparação como `df_bruto["duracao_min"] <= 0` produz `True` para as linhas que atendem à condição e `False` para as demais. Quando aplicamos `.sum()`, o Python conta os valores verdadeiros.

O símbolo `|` significa **ou** entre duas condições.


In [ ]:
# Conta distâncias que não possuem valor.
distancias_ausentes = df_bruto["distancia_km"].isna().sum()

# Cria duas condições para durações impossíveis.
duracao_negativa = df_bruto["duracao_min"] <= 0
duracao_excessiva = df_bruto["duracao_min"] > 240

# O símbolo | seleciona linhas que atendem a uma condição OU à outra.
duracoes_invalidas = (duracao_negativa | duracao_excessiva).sum()

# duplicated identifica IDs que já apareceram anteriormente.
ids_repetidos = df_bruto["evento_id"].duplicated().sum()

print("Distâncias ausentes:", distancias_ausentes)
print("Durações inválidas:", duracoes_invalidas)
print("IDs repetidos:", ids_repetidos)
print()
print("Categorias encontradas na coluna modal:")
display(df_bruto["modal"].value_counts().to_frame("quantidade"))


### Atividade — criando uma versão tratada

Preservaremos `df_bruto` e criaremos `df_tratado`. Isso permite comparar o que chegou com o que foi corrigido.

- `.copy()` cria uma cópia independente.
- `.drop_duplicates()` remove repetições.
- `.replace()` substitui uma categoria.
- `.between()` verifica se um número está dentro de um intervalo.
- `.fillna()` preenche valores ausentes.


In [ ]:
# Remove IDs repetidos e cria uma cópia independente.
df_tratado = df_bruto.drop_duplicates(subset=["evento_id"]).copy()

# Padroniza a grafia de ônibus.
df_tratado["modal"] = df_tratado["modal"].replace("Onibus", "ônibus")

# Cria um filtro: duração maior que zero e menor ou igual a 240.
duracao_valida = df_tratado["duracao_min"].between(0, 240, inclusive="neither")

# Mantém somente as linhas que passaram pelo filtro.
df_tratado = df_tratado[duracao_valida].copy()

# Calcula a mediana e a utiliza nas distâncias ausentes.
mediana_distancia = df_tratado["distancia_km"].median()
df_tratado["distancia_km"] = df_tratado["distancia_km"].fillna(mediana_distancia)

print("Linhas na camada bruta:", len(df_bruto))
print("Linhas na camada tratada:", len(df_tratado))
print("Distâncias ausentes após o tratamento:", df_tratado["distancia_km"].isna().sum())
print("IDs repetidos após o tratamento:", df_tratado["evento_id"].duplicated().sum())


### O que aprendemos com a Veracidade?

Toda regra de limpeza envolve uma decisão. Nesta aula assumimos que uma viagem não pode durar mais de 240 minutos e preenchemos distâncias com a mediana geral.

Em um projeto real, essas regras deveriam ser discutidas com especialistas, documentadas e monitoradas. Limpar dados sem compreender o contexto também pode destruir informação válida.


## 10. Valor — que decisão será melhorada?

**Valor** é o benefício produzido pelos dados. Armazenar muitos registros não garante valor. Precisamos transformar os dados em informações ligadas a uma decisão.

### Atividade — criando indicadores

`groupby("modal")` separa as linhas por meio de transporte. Em seguida, `mean()` calcula a média dentro de cada grupo.


In [ ]:
# Agrupa os dados por modal e calcula médias.
medias_por_modal = (
    df_tratado.groupby("modal")[["distancia_km", "duracao_min"]]
               .mean()
               .round(2)
)

# Conta a quantidade de eventos de cada modal.
quantidade_por_modal = df_tratado["modal"].value_counts()

# Adiciona a contagem como uma nova coluna da tabela de médias.
medias_por_modal["quantidade_eventos"] = quantidade_por_modal

display(medias_por_modal)


### Atividade — verificando o impacto da qualidade

Vamos comparar a duração média antes e depois do tratamento. Essa comparação liga Veracidade a Valor: erros de qualidade podem alterar a informação usada em uma decisão.


In [ ]:
# Calcula a média usando os dados brutos.
media_bruta = df_bruto["duracao_min"].mean()

# Calcula a média usando os dados tratados.
media_tratada = df_tratado["duracao_min"].mean()

print("Duração média nos dados brutos:", round(media_bruta, 2), "minutos")
print("Duração média nos dados tratados:", round(media_tratada, 2), "minutos")
print("Diferença causada pelos problemas:", round(media_bruta - media_tratada, 2), "minutos")


### O que aprendemos com o Valor?

Os indicadores podem ajudar a planejar capacidade, horários e oferta de transporte. Entretanto, nossos dados são sintéticos. Eles ensinam o método, mas não justificam decisões reais sobre uma cidade.

Antes de agir, a prefeitura precisaria confirmar cobertura, representatividade, privacidade e regras de qualidade.


## 11. Dos 5 Vs para uma arquitetura

Agora conseguimos justificar cada parte de uma arquitetura conceitual:

```mermaid
flowchart LR
    A[GPS] --> D[Recebimento]
    B[Bilhetagem] --> D
    C[Aplicativos e estações] --> D
    D --> E[(Dados brutos)]
    E --> F[Verificação de qualidade]
    F --> G[(Dados tratados)]
    G --> H[Indicadores rápidos]
    G --> I[Relatórios diários]
    H --> J[Operação]
    I --> K[Planejamento]
    L[Segurança, catálogo e privacidade] -.-> E
    L -.-> G
```

| Parte | Relação com os 5 Vs |
|---|---|
| Recebimento | lida com Velocidade e Variedade |
| Dados brutos | preservam o Volume recebido |
| Verificação | melhora a Veracidade |
| Dados tratados | fornecem uma base confiável |
| Indicadores e relatórios | produzem Valor |
| Segurança e catálogo | protegem e explicam os dados |

Ainda não escolhemos produtos. Primeiro entendemos os requisitos; depois selecionamos tecnologias.


## 12. Atividade integradora — resumindo as evidências

Para concluir, criaremos uma pequena tabela. Cada linha relacionará um V, uma evidência observada e uma decisão associada.

O código utiliza listas, que já vimos no início, e um dicionário, que associa nomes de colunas aos seus conteúdos.


In [ ]:
# Prepara uma lista com os nomes dos 5 Vs.
nomes = ["Volume", "Velocidade", "Variedade", "Veracidade", "Valor"]

# Prepara uma evidência para cada V.
evidencias = [
    str(len(df_bruto)) + " linhas na amostra",
    str(quantidade_no_pico) + " eventos na hora de maior movimento",
    str(df_bruto["fonte"].nunique()) + " fontes diferentes",
    str(ids_repetidos) + " IDs repetidos antes do tratamento",
    str(len(medias_por_modal)) + " modais analisados nos indicadores",
]

# Prepara uma decisão relacionada a cada evidência.
decisoes = [
    "dimensionar armazenamento",
    "definir frequência de atualização",
    "integrar e catalogar as fontes",
    "criar e monitorar regras de qualidade",
    "utilizar indicadores no planejamento",
]

# Reúne as três listas em um DataFrame.
resumo_5vs = pd.DataFrame({
    "V": nomes,
    "evidência": evidencias,
    "decisão": decisoes,
})

display(resumo_5vs)


## 13. Fechamento

Big Data é um contexto no qual a combinação de **Volume, Velocidade, Variedade, Veracidade e Valor** influencia a maneira de armazenar, processar, proteger e utilizar dados.

Nesta aula, também aprendemos os seguintes elementos de Python:

- variáveis e listas;
- importação de bibliotecas;
- criação de DataFrames;
- seleção de colunas;
- funções como `len()` e `print()`;
- métodos como `head()`, `value_counts()` e `mean()`;
- repetição com `for`;
- condições e filtros;
- agrupamento com `groupby()`.

### Perguntas para conversar com a turma

1. Qual dos 5 Vs parece mais difícil no caso de mobilidade?
2. Por que resultado calculado não significa resultado confiável?
3. O que mudaria se os dados precisassem ser analisados em segundos?
4. Por que não podemos tomar decisões reais com os dados sintéticos?
5. Em que momento uma única máquina poderia deixar de ser suficiente?

### Próxima semana

Na Semana 2, veremos como CSV, JSON e Parquet armazenam os mesmos dados de formas diferentes e como isso afeta tamanho, tipos e velocidade de leitura.
